# M001 — Supervised Pairwise Entity Matching Baseline

This notebook establishes the first learned matcher for the Amazon Business Entity Resolution Challenge.

**Model:** LightGBM binary classifier over engineered pairwise features.

**M001 deliberately does not use embeddings or an LLM.** Its purpose is to establish a clean lexical/structural baseline and reveal which signals are useful before E002 candidate generation and M002 multilingual embeddings.

**Run progression:** `5,000 S1 entities → 100,000 → full dataset`.

Every expensive stage writes intermediate output to disk so the notebook can be resumed and inspected.


In [1]:
"""

Requirements
------------
pip install pandas numpy scikit-learn rapidfuzz lightgbm

Run
---
python experiments/M001/M001_baseline_matcher.py
"""

from __future__ import annotations

import csv
import hashlib
import math
import os
import random
import re
import time
import unicodedata
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

from rapidfuzz import fuzz
from sklearn.metrics import (
    precision_score,
    recall_score,
    fbeta_score,
)
from lightgbm import LGBMClassifier, early_stopping, log_evaluation


# ============================================================
# CONFIGURATION
# ============================================================

RANDOM_SEED = 42

# Maximum number of S1 entities used for the first M001 run.
#
# Set to None for the full training set.
#
# Recommended first run:
#     100_000
#
# Once the pipeline is verified:
#     None
# Smoke test first; later use 100_000, then None.
MAX_S1_ENTITIES = 5_000

# Maximum positive pairs sampled per S1 entity.
MAX_POSITIVES_PER_ENTITY = 6

# Negative sampling.
NEGATIVE_HARD_NAME_PER_ENTITY = 3
NEGATIVE_HARD_ADDRESS_PER_ENTITY = 3
NEGATIVE_RANDOM_PER_ENTITY = 3

# Limit index buckets to prevent extremely common values
# such as "primary care" from exploding candidate counts.
MAX_BLOCK_SIZE = 500

# Number of rows used for model training/validation.
# None means use every generated pair.
MAX_TRAIN_PAIRS = None

# Validation fraction at ENTITY level.
VALIDATION_FRACTION = 0.20

# Feature calculation.
NGRAM_SIZE = 3

# LightGBM.
N_ESTIMATORS = 1000
LEARNING_RATE = 0.05
NUM_LEAVES = 31
MAX_DEPTH = -1
MIN_CHILD_SAMPLES = 50
SUBSAMPLE = 0.8
COLSAMPLE_BYTREE = 0.8

EARLY_STOPPING_ROUNDS = 75

# Threshold search.
THRESHOLD_MIN = 0.05
THRESHOLD_MAX = 0.99
THRESHOLD_STEP = 0.01

# Chunk sizes.
SOURCE_CHUNK_SIZE = 100_000
GT_CHUNK_SIZE = 100_000

# Paths.
BASE_DIR = Path(r"E:\Amazon_ML_Challenge\6ab10eb3b23ba_student_resource")
DATA_DIR = BASE_DIR /"dataset"
OUTPUT_DIR = BASE_DIR / "student_resource"/"amazon-er"/"outputs" / "M001"

S1_PATH = DATA_DIR / "train"/"train_source1.tsv"
S2_PATH = DATA_DIR / "train"/"train_source2.tsv"
S3_PATH = DATA_DIR / "train"/"train_source3.tsv"
GT_PATH = DATA_DIR / "train"/"train_ground_truth.tsv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PAIRS_PATH = OUTPUT_DIR / "M001_train_pairs.csv"
VAL_PAIRS_PATH = OUTPUT_DIR / "M001_validation_pairs.csv"
FEATURE_IMPORTANCE_PATH = OUTPUT_DIR / "M001_feature_importance.csv"
THRESHOLD_RESULTS_PATH = OUTPUT_DIR / "M001_threshold_results.csv"
MODEL_PATH = OUTPUT_DIR / "M001_model.txt"
METRICS_PATH = OUTPUT_DIR / "M001_metrics.txt"
CONFIG_PATH = OUTPUT_DIR / "M001_config.txt"
LOG_PATH = OUTPUT_DIR / "M001_progress.log"

# ============================================================


In [2]:
# LOGGING
# ============================================================

def log(message: str):
    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {message}"

    print(line, flush=True)

    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")


# ============================================================

# NORMALIZATION
# ============================================================

def normalize_text(value) -> str:
    """
    Conservative normalization.

    We intentionally DO NOT:
        - transliterate
        - remove all digits
        - delete stopwords
        - aggressively collapse business terms

    Those decisions belong to later experiments.
    """

    if value is None:
        return ""

    if pd.isna(value):
        return ""

    text = str(value)

    text = unicodedata.normalize("NFKC", text)
    text = text.casefold()

    # Convert punctuation/symbols to spaces while retaining letters
    # and digits.
    text = re.sub(r"[^\w]+", " ", text, flags=re.UNICODE)

    text = re.sub(r"\s+", " ", text).strip()

    return text


def tokenize(text: str) -> list[str]:
    if not text:
        return []

    return text.split()


def character_ngrams(text: str, n: int = NGRAM_SIZE) -> set[str]:

    if not text:
        return set()

    compact = text.replace(" ", "")

    if len(compact) < n:
        return {compact}

    return {
        compact[i:i+n]
        for i in range(len(compact) - n + 1)
    }


# ============================================================


In [3]:
# SCRIPT DETECTION
# ============================================================

def detect_script(text: str) -> str:

    if not text:
        return "Missing"

    scripts = set()

    for char in text:

        if not unicodedata.category(char).startswith("L"):
            continue

        name = unicodedata.name(char, "")

        if "LATIN" in name:
            scripts.add("Latin")
        elif "DEVANAGARI" in name:
            scripts.add("Devanagari")
        elif "BENGALI" in name:
            scripts.add("Bengali")
        elif "TELUGU" in name:
            scripts.add("Telugu")
        elif "TAMIL" in name:
            scripts.add("Tamil")
        elif "KANNADA" in name:
            scripts.add("Kannada")
        elif "MALAYALAM" in name:
            scripts.add("Malayalam")
        elif "GUJARATI" in name:
            scripts.add("Gujarati")
        elif "GURMUKHI" in name:
            scripts.add("Gurmukhi")
        elif "ARABIC" in name:
            scripts.add("Arabic")
        elif "CYRILLIC" in name:
            scripts.add("Cyrillic")
        elif "GREEK" in name:
            scripts.add("Greek")
        else:
            scripts.add("Other")

    if not scripts:
        return "NoLetters"

    return "+".join(sorted(scripts))


# ============================================================


## Data representation and ground truth

The notebook uses the challenge's actual ground-truth schema:

- `source1_entity_id`
- `matched_entity_ids`

`matched_entity_ids` contains comma-separated S2/S3 IDs.

The train/validation split is made at the **S1 entity level**, not the pair level, to prevent leakage between multiple matches belonging to the same S1 entity.


In [4]:
# RECORD REPRESENTATION
# ============================================================

class Record:

    __slots__ = (
        "entity_id",
        "name_raw",
        "address_raw",
        "country",
        "name",
        "address",
        "name_tokens",
        "address_tokens",
        "name_ngrams",
        "address_ngrams",
        "name_script",
        "address_script",
    )

    def __init__(
        self,
        entity_id: str,
        name,
        address,
        country,
    ):

        self.entity_id = str(entity_id)

        self.name_raw = "" if pd.isna(name) else str(name)
        self.address_raw = "" if pd.isna(address) else str(address)

        self.country = "" if pd.isna(country) else str(country)

        self.name = normalize_text(self.name_raw)
        self.address = normalize_text(self.address_raw)

        self.name_tokens = set(tokenize(self.name))
        self.address_tokens = set(tokenize(self.address))

        self.name_ngrams = character_ngrams(self.name)
        self.address_ngrams = character_ngrams(self.address)

        self.name_script = detect_script(self.name_raw)
        self.address_script = detect_script(self.address_raw)


# ============================================================


In [5]:
# LOAD SOURCE DATA
# ============================================================

def load_source(path: Path, source_name: str) -> dict[str, Record]:

    log("=" * 70)
    log(f"Loading {source_name}")
    log(f"Path: {path}")
    log("=" * 70)

    records = {}

    start = time.time()
    rows = 0

    for chunk_no, chunk in enumerate(
        pd.read_csv(
            path,
            sep="\t",
            dtype=str,
            chunksize=SOURCE_CHUNK_SIZE,
            keep_default_na=False,
        )
    ):

        for row in chunk.itertuples(index=False):

            records[str(row.entity_id)] = Record(
                row.entity_id,
                row.business_name,
                row.business_address,
                row.country,
            )

        rows += len(chunk)

        if chunk_no % 5 == 0:

            elapsed = time.time() - start
            rate = rows / elapsed if elapsed else 0

            log(
                f"{source_name}: "
                f"{rows:,} rows | "
                f"{rate:,.0f} rows/s"
            )

    log(
        f"{source_name} loaded: "
        f"{len(records):,} records"
    )

    return records


# ============================================================


In [6]:
# ============================================================
# GROUND TRUTH PARSING
# ============================================================

def validate_ground_truth_schema():
    sample = pd.read_csv(
        GT_PATH,
        sep="\t",
        dtype=str,
        nrows=5,
        keep_default_na=False,
    )

    expected = ["source1_entity_id", "matched_entity_ids"]

    if list(sample.columns) != expected:
        raise ValueError(
            "Unexpected ground-truth schema.\n"
            f"Expected: {expected}\n"
            f"Found:    {sample.columns.tolist()}"
        )

    log("Ground-truth schema validated.")


def parse_ground_truth_row(row):
    """
    Actual challenge schema:

        source1_entity_id
        matched_entity_ids

    matched_entity_ids is a comma-separated list of S2-* and S3-* IDs.
    """
    s1_id = str(row.source1_entity_id).strip()
    matched_value = str(row.matched_entity_ids).strip()

    if not matched_value:
        return s1_id, []

    return s1_id, [
        x.strip()
        for x in matched_value.split(",")
        if x.strip()
    ]


def load_ground_truth():
    log("=" * 70)
    log("Loading ground truth")
    log("=" * 70)

    validate_ground_truth_schema()

    gt = {}
    rows = 0

    for chunk_no, chunk in enumerate(
        pd.read_csv(
            GT_PATH,
            sep="\t",
            dtype=str,
            chunksize=GT_CHUNK_SIZE,
            keep_default_na=False,
        )
    ):
        for row in chunk.itertuples(index=False):
            s1_id, targets = parse_ground_truth_row(row)
            if s1_id:
                gt[s1_id] = set(targets)

        rows += len(chunk)

        if chunk_no % 5 == 0:
            log(
                f"GT: {rows:,} rows | "
                f"{len(gt):,} S1 entities loaded"
            )

    total_matches = sum(len(x) for x in gt.values())

    log(
        f"Ground truth loaded: "
        f"{len(gt):,} S1 entities | "
        f"{total_matches:,} positive pairs"
    )

    return gt


## Pair construction

M001 uses true matches from the ground truth and deliberately difficult negatives.

Negative types:

1. same normalized name but not a true match;
2. same normalized address but not a true match;
3. random record from the same country.

This is only a preliminary negative-generation mechanism. E002 will eventually replace it with candidates produced by the real blocking/retrieval system.


In [7]:
# BLOCK INDEX
# ============================================================

def build_block_index(
    records: dict[str, Record],
    field: str,
) -> dict[str, list[str]]:

    log(f"Building {field} block index")

    index = defaultdict(list)

    for entity_id, record in records.items():

        value = getattr(record, field)

        if not value:
            continue

        index[value].append(entity_id)

    # Remove extremely common blocks.
    filtered = {}

    for key, ids in index.items():

        if len(ids) <= MAX_BLOCK_SIZE:
            filtered[key] = ids

    log(
        f"{field} index: "
        f"{len(filtered):,} usable blocks"
    )

    return filtered


# ============================================================

# ENTITY SPLIT
# ============================================================

def create_entity_split(
    entity_ids: list[str],
) -> tuple[set[str], set[str]]:

    ids = list(entity_ids)

    rng = random.Random(RANDOM_SEED)
    rng.shuffle(ids)

    split_index = int(
        len(ids) * (1.0 - VALIDATION_FRACTION)
    )

    train_ids = set(ids[:split_index])
    validation_ids = set(ids[split_index:])

    return train_ids, validation_ids


# ============================================================


In [8]:
# NEGATIVE GENERATION
# ============================================================

def deterministic_random_choice(
    candidates: list[str],
    seed_text: str,
    count: int,
) -> list[str]:

    if not candidates:
        return []

    seed = int(
        hashlib.md5(
            seed_text.encode("utf-8")
        ).hexdigest()[:8],
        16,
    )

    rng = random.Random(seed)

    candidates = list(candidates)

    rng.shuffle(candidates)

    return candidates[:count]


def generate_negative_pairs(
    s1: Record,
    target_records: dict[str, Record],
    target_ids: list[str],
    positive_ids: set[str],
    name_index: dict[str, list[str]],
    address_index: dict[str, list[str]],
    source_name: str,
) -> list[tuple[str, str, int, str]]:

    """
    Generate hard + random negatives.

    Returns:
        (s1_id, target_id, label, negative_type)
    """

    negatives = []
    selected = set(positive_ids)

    # --------------------------------------------------------
    # Hard negatives: same normalized name
    # --------------------------------------------------------

    if s1.name:

        for target_id in name_index.get(s1.name, []):

            if target_id in selected:
                continue

            negatives.append(
                (
                    s1.entity_id,
                    target_id,
                    0,
                    "hard_name",
                )
            )

            selected.add(target_id)

            if len(negatives) >= NEGATIVE_HARD_NAME_PER_ENTITY:
                break

    # --------------------------------------------------------
    # Hard negatives: same normalized address
    # --------------------------------------------------------

    if s1.address:

        for target_id in address_index.get(s1.address, []):

            if target_id in selected:
                continue

            negatives.append(
                (
                    s1.entity_id,
                    target_id,
                    0,
                    "hard_address",
                )
            )

            selected.add(target_id)

            hard_count = sum(
                1
                for x in negatives
                if x[3] == "hard_address"
            )

            if hard_count >= NEGATIVE_HARD_ADDRESS_PER_ENTITY:
                break

    # --------------------------------------------------------
    # Random same-country negatives
    # --------------------------------------------------------

    same_country = [
        target_id
        for target_id in target_ids
        if target_id not in selected
        and target_records[target_id].country == s1.country
    ]

    random_candidates = deterministic_random_choice(
        same_country,
        s1.entity_id + source_name,
        NEGATIVE_RANDOM_PER_ENTITY,
    )

    for target_id in random_candidates:

        negatives.append(
            (
                s1.entity_id,
                target_id,
                0,
                "random_same_country",
            )
        )

    return negatives


# ============================================================

# POSITIVE / NEGATIVE PAIR CONSTRUCTION
# ============================================================

def create_pair_dataset(
    s1_records: dict[str, Record],
    target_records: dict[str, Record],
    gt: dict[str, set[str]],
    source_name: str,
    output_path: Path,
    max_s1_entities: int | None,
):

    log("=" * 70)
    log(f"Generating M001 pairs for {source_name}")
    log("=" * 70)

    name_index = build_block_index(
        target_records,
        "name",
    )

    address_index = build_block_index(
        target_records,
        "address",
    )

    valid_s1 = [
        s1_id
        for s1_id in gt
        if s1_id in s1_records
    ]

    # Stable sampling.
    rng = random.Random(RANDOM_SEED)
    rng.shuffle(valid_s1)

    if max_s1_entities is not None:
        valid_s1 = valid_s1[:max_s1_entities]

    target_ids = list(target_records.keys())

    rows = []

    for counter, s1_id in enumerate(valid_s1, start=1):

        s1 = s1_records[s1_id]

        positives = [
            target_id
            for target_id in gt.get(s1_id, set())
            if target_id in target_records
        ]

        positives = positives[:MAX_POSITIVES_PER_ENTITY]

        # Positive pairs.
        for target_id in positives:

            rows.append(
                (
                    s1_id,
                    target_id,
                    source_name,
                    1,
                    "positive",
                )
            )

        # Negative pairs.
        negatives = generate_negative_pairs(
            s1=s1,
            target_records=target_records,
            target_ids=target_ids,
            positive_ids=set(positives),
            name_index=name_index,
            address_index=address_index,
            source_name=source_name,
        )

        rows.extend(
            (
                s1_id,
                target_id,
                source_name,
                label,
                negative_type,
            )
            for s1_id, target_id, label, negative_type
            in negatives
        )

        # Incremental output.
        if len(rows) >= 50_000:

            append_pair_rows(
                output_path,
                rows,
            )

            log(
                f"{source_name}: "
                f"{counter:,}/{len(valid_s1):,} S1 entities | "
                f"pairs written incrementally"
            )

            rows.clear()

    if rows:
        append_pair_rows(output_path, rows)

    log(
        f"{source_name} pair generation complete."
    )


def append_pair_rows(
    path: Path,
    rows: list[tuple],
):

    exists = path.exists()

    with open(
        path,
        "a",
        newline="",
        encoding="utf-8",
    ) as f:

        writer = csv.writer(f)

        if not exists:
            writer.writerow([
                "s1_id",
                "target_id",
                "source",
                "label",
                "pair_type",
            ])

        writer.writerows(rows)


# ============================================================


In [9]:
# STRING SIMILARITY
# ============================================================

def safe_jaccard(
    a: set[str],
    b: set[str],
) -> float:

    if not a and not b:
        return 1.0

    if not a or not b:
        return 0.0

    union = a | b

    if not union:
        return 0.0

    return len(a & b) / len(union)


def numeric_similarity(
    a: str,
    b: str,
) -> float:

    if not a or not b:
        return 0.0

    return fuzz.ratio(a, b) / 100.0


def partial_similarity(
    a: str,
    b: str,
) -> float:

    if not a or not b:
        return 0.0

    return fuzz.partial_ratio(a, b) / 100.0


def token_sort_similarity(
    a: str,
    b: str,
) -> float:

    if not a or not b:
        return 0.0

    return fuzz.token_sort_ratio(a, b) / 100.0


def length_ratio(
    a: str,
    b: str,
) -> float:

    if not a and not b:
        return 1.0

    if not a or not b:
        return 0.0

    return min(len(a), len(b)) / max(len(a), len(b))


# ============================================================


## Pairwise feature engineering

The model combines:

- exact normalized name/address;
- token Jaccard;
- character n-gram Jaccard;
- RapidFuzz similarities;
- length similarity;
- country equality;
- missingness;
- script compatibility;
- non-ASCII indicators;
- source indicators.

Two explicit features are included because E001 showed that the **union** of exact normalized name/address is substantially more useful than either field alone:

- `name_or_address_exact`
- `name_and_address_exact`


In [10]:
# PAIR FEATURE EXTRACTION
# ============================================================

FEATURE_NAMES = [
    "country_match",

    "name_exact",
    "name_token_jaccard",
    "name_char_jaccard",
    "name_ratio",
    "name_partial_ratio",
    "name_token_sort_ratio",
    "name_length_ratio",

    "address_exact",
    "name_or_address_exact",
    "name_and_address_exact",
    "address_token_jaccard",
    "address_char_jaccard",
    "address_ratio",
    "address_partial_ratio",
    "address_token_sort_ratio",
    "address_length_ratio",

    "name_missing_left",
    "name_missing_right",
    "address_missing_left",
    "address_missing_right",
    "both_addresses_present",

    "name_script_same",
    "name_script_overlap",

    "address_script_same",
    "address_script_overlap",

    "name_non_ascii_left",
    "name_non_ascii_right",

    "address_non_ascii_left",
    "address_non_ascii_right",

    "name_length_left",
    "name_length_right",

    "address_length_left",
    "address_length_right",

    "source_s2",
    "source_s3",
]


def script_overlap(a: str, b: str) -> bool:

    if a == "Missing" or b == "Missing":
        return False

    a_set = set(a.split("+"))
    b_set = set(b.split("+"))

    return bool(a_set & b_set)


def make_features(
    left: Record,
    right: Record,
    source: str,
) -> list[float]:

    return [

        # Country
        float(left.country == right.country),

        # ---------------- NAME ----------------

        float(
            bool(left.name)
            and bool(right.name)
            and left.name == right.name
        ),

        safe_jaccard(
            left.name_tokens,
            right.name_tokens,
        ),

        safe_jaccard(
            left.name_ngrams,
            right.name_ngrams,
        ),

        numeric_similarity(
            left.name,
            right.name,
        ),

        partial_similarity(
            left.name,
            right.name,
        ),

        token_sort_similarity(
            left.name,
            right.name,
        ),

        length_ratio(
            left.name,
            right.name,
        ),

        # ---------------- ADDRESS ----------------

        float(
            bool(left.address)
            and bool(right.address)
            and left.address == right.address
        ),

        # E001 showed that the exact name/address union is an
        # important signal, so expose it explicitly.
        float(
            (
                bool(left.name)
                and bool(right.name)
                and left.name == right.name
            )
            or
            (
                bool(left.address)
                and bool(right.address)
                and left.address == right.address
            )
        ),

        float(
            (
                bool(left.name)
                and bool(right.name)
                and left.name == right.name
            )
            and
            (
                bool(left.address)
                and bool(right.address)
                and left.address == right.address
            )
        ),

        safe_jaccard(
            left.address_tokens,
            right.address_tokens,
        ),

        safe_jaccard(
            left.address_ngrams,
            right.address_ngrams,
        ),

        numeric_similarity(
            left.address,
            right.address,
        ),

        partial_similarity(
            left.address,
            right.address,
        ),

        token_sort_similarity(
            left.address,
            right.address,
        ),

        length_ratio(
            left.address,
            right.address,
        ),

        # ---------------- MISSINGNESS ----------------

        float(not left.name),
        float(not right.name),

        float(not left.address),
        float(not right.address),

        float(
            bool(left.address)
            and bool(right.address)
        ),

        # ---------------- SCRIPT ----------------

        float(
            left.name_script == right.name_script
            and left.name_script != "Missing"
        ),

        float(
            script_overlap(
                left.name_script,
                right.name_script,
            )
        ),

        float(
            left.address_script == right.address_script
            and left.address_script != "Missing"
        ),

        float(
            script_overlap(
                left.address_script,
                right.address_script,
            )
        ),

        # ---------------- NON ASCII ----------------

        float(
            any(ord(c) > 127 for c in left.name_raw)
        ),

        float(
            any(ord(c) > 127 for c in right.name_raw)
        ),

        float(
            any(ord(c) > 127 for c in left.address_raw)
        ),

        float(
            any(ord(c) > 127 for c in right.address_raw)
        ),

        # ---------------- LENGTHS ----------------

        len(left.name),
        len(right.name),

        len(left.address),
        len(right.address),

        # ---------------- SOURCE ----------------

        float(source == "S2"),
        float(source == "S3"),
    ]


# ============================================================


In [11]:
# FEATURE DATASET
# ============================================================

def create_feature_dataset(
    pair_path: Path,
    s1_records: dict[str, Record],
    s2_records: dict[str, Record],
    s3_records: dict[str, Record],
):

    log("=" * 70)
    log("Creating pairwise feature dataset")
    log("=" * 70)

    feature_path = OUTPUT_DIR / "M001_features.csv"

    if feature_path.exists():
        feature_path.unlink()

    total = 0
    start = time.time()

    for chunk_no, pairs in enumerate(
        pd.read_csv(
            pair_path,
            chunksize=50_000,
        )
    ):

        feature_rows = []

        for row in pairs.itertuples(index=False):

            left = s1_records.get(row.s1_id)

            if row.source == "S2":
                right = s2_records.get(row.target_id)
            else:
                right = s3_records.get(row.target_id)

            if left is None or right is None:
                continue

            features = make_features(
                left,
                right,
                row.source,
            )

            feature_rows.append(
                [
                    row.s1_id,
                    row.target_id,
                    row.source,
                    row.label,
                    row.pair_type,
                    *features,
                ]
            )

        columns = [
            "s1_id",
            "target_id",
            "source",
            "label",
            "pair_type",
            *FEATURE_NAMES,
        ]

        write_header = not feature_path.exists()

        with open(
            feature_path,
            "a",
            newline="",
            encoding="utf-8",
        ) as f:

            writer = csv.writer(f)

            if write_header:
                writer.writerow(columns)

            writer.writerows(feature_rows)

        total += len(feature_rows)

        elapsed = time.time() - start
        rate = total / elapsed if elapsed else 0

        log(
            f"Features: "
            f"{total:,} pairs | "
            f"{rate:,.0f} pairs/s | "
            f"elapsed {elapsed/60:.1f} min"
        )

    return feature_path


# ============================================================


## Train the first matcher

After feature generation, inspect the dataset before training.

The model is a LightGBM binary classifier. It learns:

`P(match | pair features)`

The validation threshold is selected using F0.5 because the challenge metric weights precision more heavily than recall.

This M001 score is a **diagnostic baseline**, not yet a competition-quality score, because E002 has not supplied the final candidate distribution.


In [12]:
# TRAIN / VALIDATION SPLIT
# ============================================================

def split_features(
    feature_path: Path,
):

    log("=" * 70)
    log("Creating entity-level train/validation split")
    log("=" * 70)

    df = pd.read_csv(feature_path)

    unique_s1 = df["s1_id"].unique().tolist()

    train_ids, validation_ids = create_entity_split(
        unique_s1
    )

    train_df = df[
        df["s1_id"].isin(train_ids)
    ].copy()

    val_df = df[
        df["s1_id"].isin(validation_ids)
    ].copy()

    train_df.to_csv(
        TRAIN_PAIRS_PATH,
        index=False,
    )

    val_df.to_csv(
        VAL_PAIRS_PATH,
        index=False,
    )

    log(
        f"Training: "
        f"{len(train_df):,} pairs | "
        f"{len(train_ids):,} S1 entities"
    )

    log(
        f"Validation: "
        f"{len(val_df):,} pairs | "
        f"{len(validation_ids):,} S1 entities"
    )

    log(
        f"Training positives: "
        f"{train_df.label.sum():,}"
    )

    log(
        f"Validation positives: "
        f"{val_df.label.sum():,}"
    )

    return train_df, val_df


# ============================================================

# MODEL TRAINING
# ============================================================

def train_model(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
):

    log("=" * 70)
    log("TRAINING LIGHTGBM M001")
    log("=" * 70)

    X_train = train_df[FEATURE_NAMES]
    y_train = train_df["label"].astype(int)

    X_val = val_df[FEATURE_NAMES]
    y_val = val_df["label"].astype(int)

    positive = max(int(y_train.sum()), 1)
    negative = max(int((y_train == 0).sum()), 1)

    scale_pos_weight = negative / positive

    log(
        f"Positive samples: {positive:,}"
    )

    log(
        f"Negative samples: {negative:,}"
    )

    log(
        f"Scale positive weight: "
        f"{scale_pos_weight:.3f}"
    )

    model = LGBMClassifier(
        objective="binary",

        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,

        num_leaves=NUM_LEAVES,
        max_depth=MAX_DEPTH,
        min_child_samples=MIN_CHILD_SAMPLES,

        subsample=SUBSAMPLE,
        colsample_bytree=COLSAMPLE_BYTREE,

        scale_pos_weight=scale_pos_weight,

        random_state=RANDOM_SEED,
        n_jobs=-1,

        verbosity=-1,
    )

    model.fit(
        X_train,
        y_train,

        eval_set=[
            (X_val, y_val),
        ],

        eval_metric="binary_logloss",

        callbacks=[
            early_stopping(
                EARLY_STOPPING_ROUNDS,
                first_metric_only=True,
            ),
            log_evaluation(
                period=25,
            ),
        ],
    )

    model.booster_.save_model(
        str(MODEL_PATH)
    )

    log(
        f"Best iteration: "
        f"{model.best_iteration_}"
    )

    log(
        f"Model saved to: "
        f"{MODEL_PATH}"
    )

    return model


# ============================================================


In [13]:
# FEATURE IMPORTANCE
# ============================================================

def save_feature_importance(model):

    log("=" * 70)
    log("Saving feature importance")
    log("=" * 70)

    importance = model.feature_importances_

    df = pd.DataFrame({
        "feature": FEATURE_NAMES,
        "importance": importance,
    })

    df = df.sort_values(
        "importance",
        ascending=False,
    )

    df.to_csv(
        FEATURE_IMPORTANCE_PATH,
        index=False,
    )

    for row in df.head(15).itertuples():

        log(
            f"{row.feature}: "
            f"{row.importance}"
        )


# ============================================================

# THRESHOLD EVALUATION
# ============================================================

def evaluate_thresholds(
    model,
    val_df: pd.DataFrame,
):

    log("=" * 70)
    log("THRESHOLD SEARCH")
    log("=" * 70)

    X_val = val_df[FEATURE_NAMES]
    y_val = val_df["label"].astype(int).to_numpy()

    probabilities = model.predict_proba(
        X_val,
        num_iteration=model.best_iteration_,
    )[:, 1]

    results = []

    best = None

    threshold = THRESHOLD_MIN

    while threshold <= THRESHOLD_MAX + 1e-9:

        predictions = (
            probabilities >= threshold
        ).astype(int)

        precision = precision_score(
            y_val,
            predictions,
            zero_division=0,
        )

        recall = recall_score(
            y_val,
            predictions,
            zero_division=0,
        )

        f05 = fbeta_score(
            y_val,
            predictions,
            beta=0.5,
            zero_division=0,
        )

        predicted_matches = int(
            predictions.sum()
        )

        results.append([
            threshold,
            precision,
            recall,
            f05,
            predicted_matches,
        ])

        if best is None or f05 > best["f05"]:

            best = {
                "threshold": threshold,
                "precision": precision,
                "recall": recall,
                "f05": f05,
                "predicted_matches": predicted_matches,
            }

        threshold += THRESHOLD_STEP

    results_df = pd.DataFrame(
        results,
        columns=[
            "threshold",
            "precision",
            "recall",
            "f0_5",
            "predicted_matches",
        ],
    )

    results_df.to_csv(
        THRESHOLD_RESULTS_PATH,
        index=False,
    )

    log(
        f"Best validation threshold: "
        f"{best['threshold']:.2f}"
    )

    log(
        f"Validation precision: "
        f"{best['precision']:.5f}"
    )

    log(
        f"Validation recall: "
        f"{best['recall']:.5f}"
    )

    log(
        f"Validation F0.5: "
        f"{best['f05']:.5f}"
    )

    log(
        f"Predicted positive pairs: "
        f"{best['predicted_matches']:,}"
    )

    return best, results_df


# ============================================================


In [14]:
# SAVE METRICS / CONFIG
# ============================================================

def save_metrics(
    best,
    train_df,
    val_df,
):

    with open(
        METRICS_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        f.write(
            "M001 - LightGBM Baseline Matcher\n"
        )

        f.write(
            "================================\n\n"
        )

        f.write(
            f"Best threshold: "
            f"{best['threshold']:.4f}\n"
        )

        f.write(
            f"Precision: "
            f"{best['precision']:.6f}\n"
        )

        f.write(
            f"Recall: "
            f"{best['recall']:.6f}\n"
        )

        f.write(
            f"F0.5: "
            f"{best['f05']:.6f}\n"
        )

        f.write(
            f"Predicted matches: "
            f"{best['predicted_matches']}\n\n"
        )

        f.write(
            f"Training pairs: "
            f"{len(train_df)}\n"
        )

        f.write(
            f"Validation pairs: "
            f"{len(val_df)}\n"
        )

        f.write(
            f"Training positives: "
            f"{int(train_df.label.sum())}\n"
        )

        f.write(
            f"Validation positives: "
            f"{int(val_df.label.sum())}\n"
        )


def save_config():

    with open(
        CONFIG_PATH,
        "w",
        encoding="utf-8",
    ) as f:

        f.write(
            "M001 CONFIGURATION\n"
        )

        f.write(
            "==================\n\n"
        )

        for name, value in sorted(
            globals().items()
        ):

            if name.isupper() and not name.startswith("__"):

                if isinstance(
                    value,
                    (str, int, float, bool),
                ):

                    f.write(
                        f"{name} = {value}\n"
                    )


# ============================================================

# ============================================================
# ERROR ANALYSIS
# ============================================================

def run_error_analysis(model, val_df):
    log("=" * 70)
    log("ERROR ANALYSIS")
    log("=" * 70)

    X = val_df[FEATURE_NAMES]
    y = val_df["label"].astype(int).to_numpy()

    probabilities = model.predict_proba(
        X,
        num_iteration=model.best_iteration_,
    )[:, 1]

    threshold = float(
        pd.read_csv(THRESHOLD_RESULTS_PATH)
        .sort_values("f0_5", ascending=False)
        .iloc[0]["threshold"]
    )

    result = val_df[
        ["s1_id", "target_id", "source", "label", "pair_type"]
    ].copy()

    result["probability"] = probabilities
    result["prediction"] = (
        probabilities >= threshold
    ).astype(int)

    result["error_type"] = "correct"

    result.loc[
        (result["label"] == 0) & (result["prediction"] == 1),
        "error_type",
    ] = "false_positive"

    result.loc[
        (result["label"] == 1) & (result["prediction"] == 0),
        "error_type",
    ] = "false_negative"

    error_path = OUTPUT_DIR / "M001_validation_errors.csv"

    result[result["error_type"] != "correct"].sort_values(
        "probability",
        ascending=False,
    ).to_csv(
        error_path,
        index=False,
    )

    log(f"Errors written to: {error_path}")
    log(result["error_type"].value_counts().to_string())

    return result


def print_m001_summary(train_df, val_df, best):
    log("=" * 70)
    log("M001 EXPERIMENT SUMMARY")
    log("=" * 70)

    log(f"Training pairs:       {len(train_df):,}")
    log(f"Validation pairs:     {len(val_df):,}")
    log(f"Training positives:   {int(train_df.label.sum()):,}")
    log(f"Validation positives: {int(val_df.label.sum()):,}")
    log(f"Best threshold:       {best['threshold']:.3f}")
    log(f"Precision:             {best['precision']:.6f}")
    log(f"Recall:                {best['recall']:.6f}")
    log(f"F0.5:                  {best['f05']:.6f}")

    log(
        "This is a preliminary pairwise baseline. "
        "Do not compare this F0.5 directly with the final "
        "competition score until negatives come from E002."
    )


In [15]:
# ============================================================
# RUN M001 — EXECUTE THIS LAST
# ============================================================

total_start = time.time()

save_config()

# -----------------------------
# 1. Load source records
# -----------------------------
s1_records = load_source(S1_PATH, "S1")
s2_records = load_source(S2_PATH, "S2")
s3_records = load_source(S3_PATH, "S3")

# -----------------------------
# 2. Load ground truth
# -----------------------------
gt = load_ground_truth()

available_s1 = [
    x for x in gt
    if x in s1_records
]

rng = random.Random(RANDOM_SEED)
rng.shuffle(available_s1)

if MAX_S1_ENTITIES is not None:
    available_s1 = available_s1[:MAX_S1_ENTITIES]

selected_gt = {
    x: gt[x]
    for x in available_s1
}

log(
    f"Selected {len(selected_gt):,} S1 entities "
    f"for this M001 run."
)

# -----------------------------
# 3. Generate pair datasets
# -----------------------------
s2_pair_path = OUTPUT_DIR / "M001_S2_pairs.csv"
s3_pair_path = OUTPUT_DIR / "M001_S3_pairs.csv"

# Prevent duplicate rows when rerunning the notebook.
for pair_file in [s2_pair_path, s3_pair_path]:
    if pair_file.exists():
        pair_file.unlink()

create_pair_dataset(
    s1_records=s1_records,
    target_records=s2_records,
    gt=selected_gt,
    source_name="S2",
    output_path=s2_pair_path,
    max_s1_entities=len(selected_gt),
)

create_pair_dataset(
    s1_records=s1_records,
    target_records=s3_records,
    gt=selected_gt,
    source_name="S3",
    output_path=s3_pair_path,
    max_s1_entities=len(selected_gt),
)

# -----------------------------
# 4. Combine pairs
# -----------------------------
combined_path = OUTPUT_DIR / "M001_pairs.csv"

if combined_path.exists():
    combined_path.unlink()

for i, pair_file in enumerate([s2_pair_path, s3_pair_path]):
    df = pd.read_csv(pair_file)
    df.to_csv(
        combined_path,
        mode="a",
        index=False,
        header=(i == 0),
    )

pair_preview = pd.read_csv(combined_path)

print("PAIR DATASET")
print(pair_preview.head())

print("\nPair type counts:")
print(pair_preview["pair_type"].value_counts())

print("\nSource counts:")
print(pair_preview["source"].value_counts())

# -----------------------------
# 5. Feature generation
# -----------------------------
feature_path = create_feature_dataset(
    combined_path,
    s1_records,
    s2_records,
    s3_records,
)

feature_df = pd.read_csv(feature_path)

print("\nFeature shape:", feature_df.shape)

print("\nLabel distribution:")
print(feature_df["label"].value_counts())

print("\nFeature NaN counts:")
print(
    feature_df[FEATURE_NAMES]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

# -----------------------------
# 6. Train/validation split
# -----------------------------
train_df, val_df = split_features(feature_path)

# -----------------------------
# 7. Train model
# -----------------------------
model = train_model(
    train_df,
    val_df,
)

# -----------------------------
# 8. Inspect model
# -----------------------------
save_feature_importance(model)

# -----------------------------
# 9. Threshold search
# -----------------------------
best, threshold_df = evaluate_thresholds(
    model,
    val_df,
)

# -----------------------------
# 10. Save metrics
# -----------------------------
save_metrics(
    best,
    train_df,
    val_df,
)

# -----------------------------
# 11. Error analysis
# -----------------------------
errors_df = run_error_analysis(
    model,
    val_df,
)

# -----------------------------
# 12. Final summary
# -----------------------------
print_m001_summary(
    train_df,
    val_df,
    best,
)

elapsed = time.time() - total_start

log(
    f"M001 complete. Runtime: "
    f"{elapsed / 60:.2f} minutes"
)

print("\nM001 COMPLETE")
print(f"Best validation F0.5: {best['f05']:.6f}")
print(f"Best threshold: {best['threshold']:.3f}")


[2026-09-26 13:53:38] ======================================================================
[2026-09-26 13:53:38] Loading S1
[2026-09-26 13:53:38] Path: E:\Amazon_ML_Challenge\6ab10eb3b23ba_student_resource\dataset\train\train_source1.tsv
[2026-09-26 13:53:38] ======================================================================
[2026-09-26 13:53:43] S1: 100,000 rows | 20,088 rows/s
[2026-09-26 13:54:08] S1: 600,000 rows | 19,742 rows/s
[2026-09-26 13:54:40] S1: 1,100,000 rows | 17,754 rows/s
[2026-09-26 13:55:52] S1: 1,600,000 rows | 11,925 rows/s
[2026-09-26 13:56:17] S1: 2,100,000 rows | 13,179 rows/s
[2026-09-26 13:56:22] S1 loaded: 2,206,821 records
[2026-09-26 13:56:23] ======================================================================
[2026-09-26 13:56:23] Loading S2
[2026-09-26 13:56:23] Path: E:\Amazon_ML_Challenge\6ab10eb3b23ba_student_resource\dataset\train\train_source2.tsv
[2026-09-26 13:56:23] ======================================================================
[2

MemoryError: 